In [1]:
from pathlib import Path

import pandas as pd

In [2]:
df = pd.read_csv(Path("data/2016-building-energy-benchmarking-6af870b7ba821669818300.csv"))
df.head()

,OSEBuildingID,DataYear,BuildingType,PrimaryPropertyType,PropertyName,Address,City,State,ZipCode,TaxParcelIdentificationNumber,...,Electricity(kWh),Electricity(kBtu),NaturalGas(therms),NaturalGas(kBtu),DefaultData,Comments,ComplianceStatus,Outlier,TotalGHGEmissions,GHGEmissionsIntensity
0,1,2016,NonResidential,Hotel,Mayflower park hotel,405 Olive way,Seattle,WA,98101.0,0659000030,...,1.156514e+06,3946027.0,12764.52930,1276453.0,False,NaN,Compliant,NaN,249.98,2.83
1,2,2016,NonResidential,Hotel,Paramount Hotel,724 Pine street,Seattle,WA,98101.0,0659000220,...,9.504252e+05,3242851.0,51450.81641,5145082.0,False,NaN,Compliant,NaN,295.86,2.86
2,3,2016,NonResidential,Hotel,5673-The Westin Seattle,1900 5th Avenue,Seattle,WA,98101.0,0659000475,...,1.451544e+07,49526664.0,14938.00000,1493800.0,False,NaN,Compliant,NaN,2089.28,2.19
3,5,2016,NonResidential,Hotel,HOTEL MAX,620 STEWART ST,Seattle,WA,98101.0,0659000640,...,8.115253e+05,2768924.0,18112.13086,1811213.0,False,NaN,Compliant,NaN,286.43,4.67
4,8,2016,NonResidential,Hotel,WARWICK SEATTLE HOTEL (ID8),401 LENORA ST,Seattle,WA,98121.0,0659000970,...,1.573449e+06,5368607.0,88039.98438,8803998.0,False,NaN,Compliant,NaN,505.01,2.88


## Fonction Nettoyage 

In [ ]:
BUILDING_TYPES_NON_RESIDENTIELS = [
    "NonResidential", "SPS-District K-12", "Nonresidential COS",
    "Nonresidential WA", "Campus",
]
CIBLES = ["SiteEnergyUse(kBtu)", "TotalGHGEmissions"]

def remplir_nan_largest_property(df):        # Rajout HS
    liste = df['ListOfAllPropertyUseTypes']
    largest = df['LargestPropertyUseType']
    largest_from_list = liste.split(",")[0]
    
    if pd.isna(largest):
        return largest_from_list
    else:
        return largest
    
def remplir_nan_largest_property_surface(df):   # Rajout HS
    surface = df['LargestPropertyUseTypeGFA']
    surface_tot = df['PropertyGFABuilding(s)']
    
    if pd.isna(surface):
        return surface_tot
    else:
        return surface

def nettoyage(df_raw, retirer_outliers_signales=True,verbose=True):
    
    df = df_raw.copy()


    # ---------- 1. Périmètre : non résidentiel ----------
    df = df[df["BuildingType"].isin(BUILDING_TYPES_NON_RESIDENTIELS)]

    # On retire aussi les « Multifamily » repérés via PrimaryPropertyType
    df = df[~df["PrimaryPropertyType"].str.contains("Multifamily", na=False)]


    # ---------- 2. Lignes peu fiables ----------
    # Option [A] : on suppr les outliers ou SiteEUI est < 10 pr garder grands bureaux très énergivores
    low_incoherent = (df["Outlier"] == "Low outlier") & (df["SiteEUI(kBtu/sf)"] < 20)
    df = df[~low_incoherent]
    # Option [B] : retirer tous les outliers signalés (High et Low)
    df = df[~df["Outlier"].isin(["High outlier", "Low outlier"])]

    # Retirer les lignes où DefaultData vaut True
    df = df[df["DefaultData"] != True]

    # Retirer lignes ou déclaration est incomplète 
    df = df[~df["ComplianceStatus"].isin(
        ["Missing Data", "Error - Correct Default Data"])]


    # ---------- 3. Target ----------
    # Drop les lignes ou cibles manquantes 
    df = df.dropna(subset=CIBLES)

    # Gardes ligne ou valeus de la cible > 0
    df = df[(df[CIBLES] > 0).all(axis=1)]

    #  Supprimer Outliers en Gardant les lignes où la consommation d'énergie est inférieure ou égale à 4.10^8
    df = df[df["SiteEnergyUse(kBtu)"] <= 4e8]


    # ---------- 4. Corrections de valeurs ----------
    # Majuscules + Delridge écrit de deux façons
    df["Neighborhood"] = (
        df["Neighborhood"].str.upper()
        .replace("DELRIDGE NEIGHBORHOODS", "DELRIDGE"))

    # 0 et 99 sont des erreurs de saisie -> NaN
    # (la médiane sera calculée plus tard, après le split train / test)
    df["NumberofFloors"] = df["NumberofFloors"].mask(
        df["NumberofFloors"].isin([0, 99]))

    # Nombre de bâtiments : 0 -> 1 (minimum logique)
    df.loc[df["NumberofBuildings"] == 0, "NumberofBuildings"] = 1

    # 2e et 3e usages : vide = « pas de 2e / 3e usage »
    df = df.fillna({
        "SecondLargestPropertyUseType": "Aucun",
        "SecondLargestPropertyUseTypeGFA": 0,
        "ThirdLargestPropertyUseType": "Aucun",
        "ThirdLargestPropertyUseTypeGFA": 0,})

    #LargestPropertyUseType : remplir les données manquantes par la première valeur de ListOfAllPropertyUseTypes (rajouté HS)
    df["LargestPropertyUseType"]= df.apply(remplir_nan_largest_property, axis=1)
    
    #LargestPropertyUseTypeGFA : remplir les données manquantes par PropertyGFATotal (rajouté HS)
    df["LargestPropertyUseTypeGFA"] = df.apply(remplir_nan_largest_property_surface, axis=1)
    
    # Laisser ENERGYSTARScore comme tel car volontairement NON rempli

# ---------- 5. suppression colonnes inutiles ---------- (rajouté HS)
    cols_inutiles = df.columns[df.nunique() <= 1].tolist()
    df = df.drop(columns=cols_inutiles)
    
    # ZipCode est redondant avec la latitude et la longitude, qui sont plus précises 
    # YearsENERGYSTARCertified est presque vide, mal formatée
    df = df.drop(columns=["ZipCode", "YearsENERGYSTARCertified"])
    
    return df.reset_index(drop=True)    


In [4]:
df_clean = nettoyage(df)

df_clean.head()

,OSEBuildingID,BuildingType,PrimaryPropertyType,PropertyName,Address,TaxParcelIdentificationNumber,CouncilDistrictCode,Neighborhood,Latitude,Longitude,...,SourceEUIWN(kBtu/sf),SiteEnergyUse(kBtu),SiteEnergyUseWN(kBtu),SteamUse(kBtu),Electricity(kWh),Electricity(kBtu),NaturalGas(therms),NaturalGas(kBtu),TotalGHGEmissions,GHGEmissionsIntensity
0,1,NonResidential,Hotel,Mayflower park hotel,405 Olive way,0659000030,7,DOWNTOWN,47.61220,-122.33799,...,189.000000,7226362.5,7456910.0,2003882.00,1.156514e+06,3946027.0,12764.52930,1276453.0,249.98,2.83
1,2,NonResidential,Hotel,Paramount Hotel,724 Pine street,0659000220,7,DOWNTOWN,47.61317,-122.33393,...,179.399994,8387933.0,8664479.0,0.00,9.504252e+05,3242851.0,51450.81641,5145082.0,295.86,2.86
2,3,NonResidential,Hotel,5673-The Westin Seattle,1900 5th Avenue,0659000475,7,DOWNTOWN,47.61393,-122.33810,...,244.100006,72587024.0,73937112.0,21566554.00,1.451544e+07,49526664.0,14938.00000,1493800.0,2089.28,2.19
3,5,NonResidential,Hotel,HOTEL MAX,620 STEWART ST,0659000640,7,DOWNTOWN,47.61412,-122.33664,...,224.000000,6794584.0,6946800.5,2214446.25,8.115253e+05,2768924.0,18112.13086,1811213.0,286.43,4.67
4,8,NonResidential,Hotel,WARWICK SEATTLE HOTEL (ID8),401 LENORA ST,0659000970,7,DOWNTOWN,47.61375,-122.34047,...,215.600006,14172606.0,14656503.0,0.00,1.573449e+06,5368607.0,88039.98438,8803998.0,505.01,2.88


In [5]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1542 entries, 0 to 1541
Data columns (total 37 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   OSEBuildingID                    1542 non-null   int64  
 1   BuildingType                     1542 non-null   str    
 2   PrimaryPropertyType              1542 non-null   str    
 3   PropertyName                     1542 non-null   str    
 4   Address                          1542 non-null   str    
 5   TaxParcelIdentificationNumber    1542 non-null   str    
 6   CouncilDistrictCode              1542 non-null   int64  
 7   Neighborhood                     1542 non-null   str    
 8   Latitude                         1542 non-null   float64
 9   Longitude                        1542 non-null   float64
 10  YearBuilt                        1542 non-null   int64  
 11  NumberofBuildings                1542 non-null   float64
 12  NumberofFloors                 

In [6]:
# Enregistrement de df_clean
df_clean.to_csv("data/df_clean.csv", index = False)